# VALENCE v0.7.0 — One-Click Colab Release

This notebook contains the public-ready VALENCE v0.7.0 source snapshot.

It automatically:

- mounts `MyDrive/valence`;
- installs the release;
- runs all 64 automated tests;
- runs healthy, regional-outage, and checkpoint-finality demonstrations;
- generates the poster-capability summary; and
- saves source, results, test reports, and run hashes to Google Drive.

Choose **Runtime → Run all**.


In [ ]:
# STEP 1 — Mount Drive and define persistent locations.
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source"
DRIVE_ARCHIVE = DRIVE_ROOT / "archive"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.7.0"
DRIVE_TEST_RESULTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_7_workspace")
RUNTIME_PROJECT = WORK_ROOT / "valence-public-v0.7.0"

for folder in (DRIVE_ROOT, DRIVE_ARCHIVE, DRIVE_RESULTS, DRIVE_TEST_RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

print(f"VALENCE root: {DRIVE_ROOT}")


In [ ]:
# STEP 2 — Reconstruct the complete public v0.7.0 repository.

import base64
import zipfile

ARCHIVE_B64 = """
...
AAtSAAD0wA8AAAA=
""".replace("\n", "").strip()

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

embedded_zip = WORK_ROOT / "valence-public-v0.7.0.zip"
embedded_zip.write_bytes(base64.b64decode(ARCHIVE_B64))

with zipfile.ZipFile(embedded_zip, "r") as archive:
    archive.extractall(WORK_ROOT)

PROJECT_ROOT = RUNTIME_PROJECT
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Embedded VALENCE v0.7.0 source did not extract correctly.")

backup = DRIVE_ARCHIVE / "source-before-v0.7.0"
if DRIVE_SOURCE.exists() and not backup.exists():
    shutil.copytree(DRIVE_SOURCE, backup)
    print(f"Previous source archived at: {backup}")

if DRIVE_SOURCE.exists():
    shutil.rmtree(DRIVE_SOURCE)
shutil.copytree(PROJECT_ROOT, DRIVE_SOURCE)
shutil.copy2(embedded_zip, DRIVE_ROOT / "valence-public-v0.7.0.zip")

print(f"Runtime source: {PROJECT_ROOT}")
print(f"Drive source: {DRIVE_SOURCE}")


In [ ]:
# RESUME GUARD — reconstruct paths after a Colab runtime reset.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source"
DRIVE_ARCHIVE = DRIVE_ROOT / "archive"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.7.0"
DRIVE_TEST_RESULTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_7_workspace")
RUNTIME_PROJECT = WORK_ROOT / "valence-public-v0.7.0"
PROJECT_ROOT = RUNTIME_PROJECT if (RUNTIME_PROJECT / "pyproject.toml").exists() else DRIVE_SOURCE

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to create the VALENCE source tree.")

for folder in (DRIVE_ROOT, DRIVE_ARCHIVE, DRIVE_RESULTS, DRIVE_TEST_RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    if not existing_pythonpath
    else str(PROJECT_ROOT / "src") + os.pathsep + existing_pythonpath
)

print(f"Resuming with project source: {PROJECT_ROOT}")

# STEP 3 — Install VALENCE and run all automated tests.
import xml.etree.ElementTree as ET

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-build-isolation",
        "-e",
        f"{PROJECT_ROOT}[dev]",
    ],
    check=True,
)

text_report = DRIVE_TEST_RESULTS / "pytest-v0.7.0-output.txt"
junit_report = DRIVE_TEST_RESULTS / "pytest-v0.7.0-junit.xml"

started = time.perf_counter()
completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "--junitxml",
        str(junit_report),
        str(PROJECT_ROOT / "tests"),
    ],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
elapsed = time.perf_counter() - started
report = completed.stdout + ("\n" + completed.stderr if completed.stderr else "")
report += f"\nElapsed seconds: {elapsed:.3f}\n"
text_report.write_text(report, encoding="utf-8")
print(report)

if completed.returncode != 0:
    raise RuntimeError(f"Tests failed. Report: {text_report}")

root = ET.parse(junit_report).getroot()
suites = [root] if root.tag.endswith("testsuite") else list(root.findall("./testsuite"))
tests_run = sum(int(s.attrib.get("tests", 0)) for s in suites)
failures = sum(int(s.attrib.get("failures", 0)) for s in suites)
errors = sum(int(s.attrib.get("errors", 0)) for s in suites)
skipped = sum(int(s.attrib.get("skipped", 0)) for s in suites)

print(f"JUnit results: tests={tests_run}, failures={failures}, errors={errors}, skipped={skipped}")
if tests_run < 64 or failures or errors:
    raise RuntimeError("The v0.7.0 test oracle did not pass.")


In [ ]:
# RESUME GUARD — reconstruct paths after a Colab runtime reset.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source"
DRIVE_ARCHIVE = DRIVE_ROOT / "archive"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.7.0"
DRIVE_TEST_RESULTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_7_workspace")
RUNTIME_PROJECT = WORK_ROOT / "valence-public-v0.7.0"
PROJECT_ROOT = RUNTIME_PROJECT if (RUNTIME_PROJECT / "pyproject.toml").exists() else DRIVE_SOURCE

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to create the VALENCE source tree.")

for folder in (DRIVE_ROOT, DRIVE_ARCHIVE, DRIVE_RESULTS, DRIVE_TEST_RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    if not existing_pythonpath
    else str(PROJECT_ROOT / "src") + os.pathsep + existing_pythonpath
)

print(f"Resuming with project source: {PROJECT_ROOT}")

# STEP 4 — Run the healthy availability oracle twice.
SMOKE_OUTPUT = DRIVE_RESULTS / "smoke"
REPEAT_OUTPUT = Path("/content/valence-v0.7-smoke-repeat")

for path in (SMOKE_OUTPUT, REPEAT_OUTPUT):
    if path.exists():
        shutil.rmtree(path)

command = ["valence", "run", str(PROJECT_ROOT / "configs" / "smoke.yaml")]
subprocess.run(command + ["--output", str(SMOKE_OUTPUT)], cwd=PROJECT_ROOT, check=True)
subprocess.run(command + ["--output", str(REPEAT_OUTPUT)], cwd=PROJECT_ROOT, check=True)

first_hash = (SMOKE_OUTPUT / "run_hash.txt").read_text().strip()
second_hash = (REPEAT_OUTPUT / "run_hash.txt").read_text().strip()
summary = json.loads((SMOKE_OUTPUT / "summary.json").read_text())

print(f"First hash:  {first_hash}")
print(f"Second hash: {second_hash}")
print(f"Proposal availability: {summary['proposal_availability']:.4f}")
print(f"Attestation availability: {summary['attestation_availability']:.4f}")
print(f"Fork episodes: {summary['forks']['fork_episode_count']}")

if first_hash != second_hash:
    raise AssertionError("Determinism check failed.")
if summary["proposal_availability"] != 1.0 or summary["attestation_availability"] != 1.0:
    raise AssertionError("Healthy availability oracle failed.")


In [ ]:
# RESUME GUARD — reconstruct paths after a Colab runtime reset.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source"
DRIVE_ARCHIVE = DRIVE_ROOT / "archive"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.7.0"
DRIVE_TEST_RESULTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_7_workspace")
RUNTIME_PROJECT = WORK_ROOT / "valence-public-v0.7.0"
PROJECT_ROOT = RUNTIME_PROJECT if (RUNTIME_PROJECT / "pyproject.toml").exists() else DRIVE_SOURCE

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to create the VALENCE source tree.")

for folder in (DRIVE_ROOT, DRIVE_ARCHIVE, DRIVE_RESULTS, DRIVE_TEST_RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    if not existing_pythonpath
    else str(PROJECT_ROOT / "src") + os.pathsep + existing_pythonpath
)

print(f"Resuming with project source: {PROJECT_ROOT}")

# STEP 5 — Run the explicit regional-outage and recovery oracle.
OUTAGE_OUTPUT = DRIVE_RESULTS / "outage-demo"
if OUTAGE_OUTPUT.exists():
    shutil.rmtree(OUTAGE_OUTPUT)

subprocess.run(
    [
        "valence",
        "run",
        str(PROJECT_ROOT / "configs" / "outage_demo.yaml"),
        "--output",
        str(OUTAGE_OUTPUT),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

outage = json.loads((OUTAGE_OUTPUT / "summary.json").read_text())
print(f"Affected validators: {outage['faults'][0]['target_count']}")
print(f"Affected stake: {outage['faults'][0]['target_stake']:.4f}")
print(f"Attestation availability: {outage['attestation_availability']:.4f}")
print(f"Missed attestation duties: {outage['attestation_duties_missed_offline']}")
print(f"Fork episodes: {outage['forks']['fork_episode_count']}")
print(f"Orphaned blocks: {outage['forks']['orphaned_blocks']}")
print(f"Maximum reorganization depth: {outage['maximum_reorganization_depth']}")

if outage["attestation_duties_missed_offline"] != 8:
    raise AssertionError("Outage missed-duty oracle failed.")
if set(outage["validator_status"].values()) != {"online"}:
    raise AssertionError("Recovery oracle failed.")


In [ ]:
# RESUME GUARD — reconstruct paths after a Colab runtime reset.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source"
DRIVE_ARCHIVE = DRIVE_ROOT / "archive"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.7.0"
DRIVE_TEST_RESULTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_7_workspace")
RUNTIME_PROJECT = WORK_ROOT / "valence-public-v0.7.0"
PROJECT_ROOT = RUNTIME_PROJECT if (RUNTIME_PROJECT / "pyproject.toml").exists() else DRIVE_SOURCE

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to create the VALENCE source tree.")

for folder in (DRIVE_ROOT, DRIVE_ARCHIVE, DRIVE_RESULTS, DRIVE_TEST_RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    if not existing_pythonpath
    else str(PROJECT_ROOT / "src") + os.pathsep + existing_pythonpath
)

print(f"Resuming with project source: {PROJECT_ROOT}")

# STEP 6 — Run checkpoint time-to-finality with right censoring.
FINALITY_OUTPUT = DRIVE_RESULTS / "finality-demo"
if FINALITY_OUTPUT.exists():
    shutil.rmtree(FINALITY_OUTPUT)

subprocess.run(
    [
        "valence",
        "run",
        str(PROJECT_ROOT / "configs" / "finality_demo.yaml"),
        "--output",
        str(FINALITY_OUTPUT),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

finality = json.loads((FINALITY_OUTPUT / "summary.json").read_text())
timing = finality["finality_timing"]
print(f"Finalized checkpoints: {timing['finalized_checkpoints']}")
print(f"Right-censored checkpoints: {timing['censored_checkpoints']}")
print(f"p95 checkpoint TTF: {timing['p95_checkpoint_time_to_finality_ms']:.0f} ms")

if timing["finalized_checkpoints"] != 2 or timing["censored_checkpoints"] != 1:
    raise AssertionError("Checkpoint finality-timing oracle failed.")


In [ ]:
# RESUME GUARD — reconstruct paths after a Colab runtime reset.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source"
DRIVE_ARCHIVE = DRIVE_ROOT / "archive"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.7.0"
DRIVE_TEST_RESULTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_7_workspace")
RUNTIME_PROJECT = WORK_ROOT / "valence-public-v0.7.0"
PROJECT_ROOT = RUNTIME_PROJECT if (RUNTIME_PROJECT / "pyproject.toml").exists() else DRIVE_SOURCE

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to create the VALENCE source tree.")

for folder in (DRIVE_ROOT, DRIVE_ARCHIVE, DRIVE_RESULTS, DRIVE_TEST_RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    if not existing_pythonpath
    else str(PROJECT_ROOT / "src") + os.pathsep + existing_pythonpath
)

print(f"Resuming with project source: {PROJECT_ROOT}")

# STEP 7 — Generate the poster-capability summary.
POSTER_OUTPUT = DRIVE_RESULTS / "poster-compliance"
if POSTER_OUTPUT.exists():
    shutil.rmtree(POSTER_OUTPUT)

subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_poster_compliance_demo.py"),
        "--output",
        str(POSTER_OUTPUT),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

poster = json.loads((POSTER_OUTPUT / "poster_capability_summary.json").read_text())
print(json.dumps(poster, indent=2, sort_keys=True))


In [ ]:
# RESUME GUARD — reconstruct paths after a Colab runtime reset.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source"
DRIVE_ARCHIVE = DRIVE_ROOT / "archive"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.7.0"
DRIVE_TEST_RESULTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_7_workspace")
RUNTIME_PROJECT = WORK_ROOT / "valence-public-v0.7.0"
PROJECT_ROOT = RUNTIME_PROJECT if (RUNTIME_PROJECT / "pyproject.toml").exists() else DRIVE_SOURCE

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to create the VALENCE source tree.")

for folder in (DRIVE_ROOT, DRIVE_ARCHIVE, DRIVE_RESULTS, DRIVE_TEST_RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PROJECT_ROOT / "src")
    if not existing_pythonpath
    else str(PROJECT_ROOT / "src") + os.pathsep + existing_pythonpath
)

print(f"Resuming with project source: {PROJECT_ROOT}")

# STEP 8 — Confirm persistent outputs.
print("VALENCE v0.7.0 completed successfully.\n")
print(f"Current source:       {DRIVE_SOURCE}")
print(f"Source archive:       {DRIVE_ROOT / 'valence-public-v0.7.0.zip'}")
print(f"Test reports:         {DRIVE_TEST_RESULTS}")
print(f"Healthy results:      {DRIVE_RESULTS / 'smoke'}")
print(f"Outage results:       {DRIVE_RESULTS / 'outage-demo'}")
print(f"Finality results:     {DRIVE_RESULTS / 'finality-demo'}")
print(f"Poster summary:       {DRIVE_RESULTS / 'poster-compliance'}")
